# Dataset Loading

In [1074]:
import json
import pandas as pd

database = "../data/raw_credit_applications.json"

with open(database, "r", encoding="utf-8") as f:
    raw = json.load(f)

type(raw), len(raw)


(list, 502)

In [1075]:
print(raw[0].keys()) # checking what fields exist at the top level

print(raw[0]) # checking the first record of the database fully

dict_keys(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp'])
{'_id': 'app_200', 'applicant_info': {'full_name': 'Jerry Smith', 'email': 'jerry.smith17@hotmail.com', 'ssn': '596-64-4340', 'ip_address': '192.168.48.155', 'gender': 'Male', 'date_of_birth': '2001-03-09', 'zip_code': '10036'}, 'financials': {'annual_income': 73000, 'credit_history_months': 23, 'debt_to_income': 0.2, 'savings_balance': 31212}, 'spending_behavior': [{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790}, {'category': 'Alcohol', 'amount': 247}], 'decision': {'loan_approved': False, 'rejection_reason': 'algorithm_risk_score'}, 'processing_timestamp': '2024-01-15T00:00:00Z'}


In [1076]:
df = pd.json_normalize(raw, sep=".") # converting the nested JSON into a pandas dataframe


df.head(3)

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


By taking a first look at the dataset we can see that the spending_behaviour column is different from the other ones. It is a list of dictionaries so we need to adress that. What I will do now is transform the list of dictionaries into separate columns


In [1077]:
# Step 1: explode list
sp = df[["_id", "spending_behavior"]].explode("spending_behavior")

# Step 2: normalize dictionaries
sp_norm = pd.json_normalize(sp["spending_behavior"])
sp_norm.index = sp.index

sp = pd.concat([sp[["_id"]], sp_norm], axis=1)

# Step 3: ensure numeric
sp["amount"] = pd.to_numeric(sp["amount"], errors="coerce")

# Step 4: pivot to wide format
sp_pivot = (
    sp.pivot_table(
        index="_id",
        columns="category",
        values="amount",
        aggfunc="sum",
        fill_value=0
    )
    .add_prefix("spend_")
    .reset_index()
)

# Step 5: merge back to main df
df = df.drop(columns=["spending_behavior"]).merge(sp_pivot, on="_id", how="left")

In [1078]:
df.head(3)

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0


In [1079]:
df.columns

Index(['_id', 'processing_timestamp', 'applicant_info.full_name',
       'applicant_info.email', 'applicant_info.ssn',
       'applicant_info.ip_address', 'applicant_info.gender',
       'applicant_info.date_of_birth', 'applicant_info.zip_code',
       'financials.annual_income', 'financials.credit_history_months',
       'financials.debt_to_income', 'financials.savings_balance',
       'decision.loan_approved', 'decision.rejection_reason', 'loan_purpose',
       'decision.interest_rate', 'decision.approved_amount',
       'financials.annual_salary', 'notes', 'spend_Adult Entertainment',
       'spend_Alcohol', 'spend_Dining', 'spend_Education',
       'spend_Entertainment', 'spend_Fitness', 'spend_Gambling',
       'spend_Groceries', 'spend_Healthcare', 'spend_Insurance', 'spend_Rent',
       'spend_Shopping', 'spend_Transportation', 'spend_Travel',
       'spend_Utilities'],
      dtype='object')

# Missing Values Analysis



We will now conduct the missing values analysis. As a missing value we mean NaN or blank text

In [1080]:
import numpy as np

# Replace empty or whitespace-only strings with NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

missing_counts = df.isna().sum()
missing_percent = (df.isna().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
}).sort_values("missing_count", ascending=False)

missing_summary

,missing_count,missing_percent
notes,500,99.60
financials.annual_salary,497,99.00
loan_purpose,452,90.04
processing_timestamp,440,87.65
decision.rejection_reason,292,58.17
decision.approved_amount,210,41.83
decision.interest_rate,210,41.83
applicant_info.email,7,1.39
applicant_info.date_of_birth,5,1.00
financials.annual_income,5,1.00


The decision.rejection_reason variable has a 58% missing rate which is quite high. However this is a variable that should only have text if the loan_approved = False. So we will now check if there are cases where that is not the case. If the output of the following code is 0 then this means this part of the dataset is consistent.


In [1081]:
# rejection_reason should exist only if loan_approved == False
invalid_rejection = df[
    (df["decision.loan_approved"] == True) &
    (df["decision.rejection_reason"].notna())
]

len(invalid_rejection)

0

Checking if there are cases where the loan was not approved without any justification. If the ouput of the code is 0 it means that all rejected loans have a justification

In [1082]:
invalid_missing_reason = df[
    (df["decision.loan_approved"] == False) &
    (df["decision.rejection_reason"].isna())
]

len(invalid_missing_reason)

0

Checking if there are cases where the loan was not approved but amount or interest columns have values which should not happen

In [1083]:
invalid_approved_fields = df[
    (df["decision.loan_approved"] == False) &
    (
        df["decision.approved_amount"].notna() |
        df["decision.interest_rate"].notna()
    )
]

len(invalid_approved_fields)

0

Checking if there are cases where the loan was approved but there are missing values in the amount or interest column

In [1084]:
invalid_missing_approved_fields = df[
    (df["decision.loan_approved"] == True) &
    (
        df["decision.approved_amount"].isna() |
        df["decision.interest_rate"].isna()
    )
]

len(invalid_missing_approved_fields)

0

# Duplicate Values Analysis


Total number of rows before any filtering:

In [1085]:
rows_before = df.shape[0]
print("Rows before duplicate removal:", rows_before)

Rows before duplicate removal: 502


In [1086]:
df["_id"].duplicated().sum()

2

These are the rows that have the same application id. 

In [1087]:
pd.set_option("display.max_columns", None) # code to show all columns

By checking the Notes column we can see that for one of the duplicated entries the system detected a duplicate submission and for the other the applicant resubmitted the application.

In [1088]:
df[df["_id"].duplicated(keep=False)].sort_values("_id") 


,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
383,app_001,NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,102000,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,1152,0,0,0,0,0,0,0,0,0
455,app_001,NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,NaN,NaN,NaN,NaN,NaN,102000,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR,0,0,0,0,0,1152,0,0,0,0,0,0,0,0,0
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0
354,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,RESUBMISSION,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0


For app_001 duplicate. I removed 455 because it less complete with information than row 383.

For app_042 duplicate, the rows have the exact same information except for the 'Notes' column so I removed row 354 which has the Note saying it was a 'Resubmition'.

In [1089]:
df = df.drop(index=[455, 354])

df.loc[[8, 383]]



,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0
383,app_001,NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,102000,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,1152,0,0,0,0,0,0,0,0,0


If there is the same SSN more than 1 time this means that the same person applied more than once.

In [1090]:
df["applicant_info.ssn"].duplicated().sum()

5

In [1091]:
ssn_col = "applicant_info.ssn"

# 1) SSN duplicates excluding NaN SSNs
df_ssn = df[df[ssn_col].notna()].copy()
n_dup_ssn_extra = df_ssn[ssn_col].duplicated().sum()
n_dup_ssn_rows = df_ssn[df_ssn[ssn_col].duplicated(keep=False)].shape[0]


# 2) Show the duplicate SSN groups with key identity fields
df_flagged = (
    df_ssn[df_ssn[ssn_col].duplicated(keep=False)]
    .sort_values(ssn_col)[["_id","applicant_info.full_name","applicant_info.gender","applicant_info.date_of_birth",ssn_col]]
)
df_flagged

,_id,applicant_info.full_name,applicant_info.gender,applicant_info.date_of_birth,applicant_info.ssn
92,app_088,Susan Martinez,F,1986-10-15,780-24-9300
122,app_016,Gary Wilson,M,1959-12-11,780-24-9300
16,app_101,Sandra Smith,Female,1997-03-23,937-72-8731
499,app_234,Samuel Hill,Male,1976/01/29,937-72-8731


- Case 1: SSN: 780-24-9300

app_088 → Susan Martinez (F, 1986-10-15)

app_016 → Gary Wilson (M, 1959-12-11)

- Case 2: SSN: 937-72-8731

app_101 → Sandra Smith (Female, 1997-03-23)

app_234 → Samuel Hill (Male, 1976/01/29)







In [1092]:
# Initialize flagged records tracker
df_flagged = pd.DataFrame(columns=df.columns.tolist() + ["flag_reason"])

The SSN duplicate analysis revealed cases where the same SSN was associated with different identities, which represents a serious data validity issue. These records are added to a new dataframe called df_flagged and removed from the our now df_clean dataset.

In [1093]:
df_flagged = pd.concat([df_flagged, df[df["_id"].isin(["app_088", "app_016", "app_101", "app_234"])].assign(flag_reason="SSN collision")])
df_clean = df[~df["_id"].isin(["app_088", "app_016", "app_101", "app_234"])].copy()

C:\Users\fmdmn\AppData\Local\Temp\ipykernel_6220\768943850.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_flagged = pd.concat([df_flagged, df[df["_id"].isin(["app_088", "app_016", "app_101", "app_234"])].assign(flag_reason="SSN collision")])


In [1094]:
df_flagged

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,flag_reason
16,app_101,NaN,Sandra Smith,sandra.smith99@icloud.com,937-72-8731,172.30.237.98,Female,1997-03-23,90255,55000,0,0.30,15364,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,464,0,0,527,0,0,0,0,0,0,0,SSN collision
92,app_088,NaN,Susan Martinez,susan.martinez83@protonmail.com,780-24-9300,172.23.224.13,F,1986-10-15,90229,55000,55,0.43,29118,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,339,0,0,SSN collision
122,app_016,NaN,Gary Wilson,gary.wilson85@yahoo.com,780-24-9300,192.168.20.42,M,1959-12-11,10091,123000,71,0.37,86599,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,485,0,0,0,0,0,0,0,SSN collision
499,app_234,NaN,Samuel Hill,samuel.hill67@protonmail.com,937-72-8731,10.143.146.157,Male,1976/01/29,10090,96000,60,0.30,38703,False,algorithm_risk_score,education,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,526,0,0,0,0,0,SSN collision


After excluding missing values, no duplicate email addresses were identified. The apparent duplicates were due to multiple missing email entries, which represent completeness issues rather than identity collisions.

In [1095]:
df[df["applicant_info.email"].notna()]["applicant_info.email"].duplicated().sum()

0

Checking if there are entries with the same Full Name + Date of Birth:

In [1096]:
df.duplicated(  
    subset=["applicant_info.full_name", "applicant_info.date_of_birth"]
).sum()

0

### Up to this point, 6 rows were removed from the original dataset. 4 of them were added to df_flagged (the other 2 weren´t because the notes column gave an explanation to the errors encountered in them) So df_clean as of now has 502-6= 496 rows

In [1097]:
len(df_clean)

496

In [1098]:
len(df_flagged)

4

In [1099]:
df_flagged

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,flag_reason
16,app_101,NaN,Sandra Smith,sandra.smith99@icloud.com,937-72-8731,172.30.237.98,Female,1997-03-23,90255,55000,0,0.30,15364,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,464,0,0,527,0,0,0,0,0,0,0,SSN collision
92,app_088,NaN,Susan Martinez,susan.martinez83@protonmail.com,780-24-9300,172.23.224.13,F,1986-10-15,90229,55000,55,0.43,29118,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,339,0,0,SSN collision
122,app_016,NaN,Gary Wilson,gary.wilson85@yahoo.com,780-24-9300,192.168.20.42,M,1959-12-11,10091,123000,71,0.37,86599,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,485,0,0,0,0,0,0,0,SSN collision
499,app_234,NaN,Samuel Hill,samuel.hill67@protonmail.com,937-72-8731,10.143.146.157,Male,1976/01/29,10090,96000,60,0.30,38703,False,algorithm_risk_score,education,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,526,0,0,0,0,0,SSN collision


# Standardize Data Types

Standardize gender (Male/Female/Unknown):

In [1100]:
# 1) Normalize gender values into a consistent set: Male, Female, Unknown
gender_map = {
    "M": "Male",
    "MALE": "Male",
    "F": "Female",
    "FEMALE": "Female"
}

df_clean["applicant_info.gender"] = (
    df_clean["applicant_info.gender"]
      .astype("string")
      .str.strip()
      .str.upper()
      .map(gender_map)  # maps M/F/MALE/FEMALE -> Male/Female
      .fillna(df_clean["applicant_info.gender"].astype("string").str.strip())  # keep already-good values
)

df_clean["applicant_info.gender"] = df_clean["applicant_info.gender"].replace({"": pd.NA}).fillna("Unknown")

Standardize Date of Birth column format:

In [1101]:
# Standardize DOB from multiple formats into a single datetime column (NaT if truly invalid)

s = (
    df_clean["applicant_info.date_of_birth"]
      .astype("string")
      .str.strip()
      .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
)

# 1) Unambiguous formats first
d_ymd_dash  = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")  # 2001-03-09
d_ymd_slash = pd.to_datetime(s, errors="coerce", format="%Y/%m/%d")  # 1990/07/26

# 2) Slash ambiguous: try both
d_dmy_slash = pd.to_datetime(s, errors="coerce", format="%d/%m/%Y")  # 14/02/1982
d_mdy_slash = pd.to_datetime(s, errors="coerce", format="%m/%d/%Y")  # 03/20/1968

# 3) Dash ambiguous: try both (then choose using >12 rule)
mask_dash = s.str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
part1 = pd.to_numeric(s.str.split("-", expand=True)[0], errors="coerce")  # first chunk

d_dmy_dash = pd.to_datetime(s.where(mask_dash), errors="coerce", format="%d-%m-%Y")
d_mdy_dash = pd.to_datetime(s.where(mask_dash), errors="coerce", format="%m-%d-%Y")
d_dash = d_dmy_dash.where(part1 > 12, d_mdy_dash)

# 4) Combine (priority order)
df_clean["applicant_info.date_of_birth"] = (
    d_ymd_dash
      .fillna(d_ymd_slash)
      .fillna(d_dmy_slash)
      .fillna(d_mdy_slash)
      .fillna(d_dash)
)

In [1102]:
df_clean.head(10)

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,247,0,0,0,0,0,0,0,0,790,480,0,0,0
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,96,0,0,0,0,0,243,0,608,0,0,0,0
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN,0,0,0,0,0,0,0,0,0,0,109,0,0,0,0
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN,0,0,0,0,0,575,0,0,0,0,0,0,0,0,0
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,463,0,0,0,0,0,0,0,0,0,0
5,app_275,NaN,Maria Miller,maria.miller67@outlook.com,417-25-4912,172.25.58.70,Female,1982-02-14,10019,110000,33,0.05,49933,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,571,0,0,0,0,0,0,0,0,0,0
6,app_099,NaN,Nicholas King,nicholas.king46@outlook.com,613-23-2503,10.62.62.45,Male,1990-01-28,10022,55000,61,0.17,30159,True,NaN,NaN,5.6,27000.0,NaN,NaN,0,0,458,0,0,0,0,0,0,0,0,0,0,0,0
7,app_246,NaN,Susan Rivera,susan.rivera74@gmail.com,176-97-1864,192.168.158.59,Female,1991-10-11,90223,82000,31,0.29,21809,True,NaN,auto,2.8,38000.0,NaN,NaN,0,0,0,0,0,0,0,0,478,0,0,0,0,0,0
8,app_042,NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,69000,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,936,0,0,0,0,0,0,306,0,0,0,0,0
9,app_348,NaN,Michael Mitchell,michael.mitchell42@hotmail.com,100-94-8400,172.28.12.121,Male,1989-10-10,10080,55000,5,0.41,13794,False,insufficient_credit_history,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,199,0,0,0,0,0,0,0,851,0


Zip Codes should be considered strings:

In [1103]:
df_clean["applicant_info.zip_code"].map(type).value_counts()

applicant_info.zip_code
<class 'str'>      495
<class 'float'>      1
Name: count, dtype: int64

In [1104]:
df_clean.loc[
    df_clean["applicant_info.zip_code"].map(type) == float,
    ["_id", "applicant_info.zip_code"]
]

,_id,applicant_info.zip_code
26,app_075,NaN


We standardize all zip code records into panda strings, which automatically makes the NaN record a pd.NaN

In [1105]:
df_clean["applicant_info.zip_code"] = (
    df_clean["applicant_info.zip_code"]
        .astype("string")
        .str.strip()
)

Standardize processing_timestamp into datetime:

In [1106]:
df_clean["processing_timestamp"] = pd.to_datetime(
    df_clean["processing_timestamp"],
    errors="coerce",
    utc=True
)

Standardizing annual_income to numeric:

In [1107]:
df_clean["financials.annual_income"] = pd.to_numeric(
    df_clean["financials.annual_income"],
    errors="coerce"
)

These columns should all be treated as text:

In [1108]:
text_cols = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "loan_purpose",
    "notes",
    "decision.rejection_reason"
]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
                .astype("string")
                .str.strip()
                .replace({"": pd.NA})
        )

In [1109]:
df_clean.dtypes

_id                                              object
processing_timestamp                datetime64[ns, UTC]
applicant_info.full_name                 string[python]
applicant_info.email                     string[python]
applicant_info.ssn                       string[python]
applicant_info.ip_address                string[python]
applicant_info.gender                            object
applicant_info.date_of_birth             datetime64[ns]
applicant_info.zip_code                  string[python]
financials.annual_income                        float64
financials.credit_history_months                  int64
financials.debt_to_income                       float64
financials.savings_balance                        int64
decision.loan_approved                             bool
decision.rejection_reason                string[python]
loan_purpose                             string[python]
decision.interest_rate                          float64
decision.approved_amount                        

# Data Validity and Cohesion

Income and savings should never be negative.

In [1110]:
df_clean.loc[
    (df_clean["financials.annual_income"] < 0) |
    (df_clean["financials.savings_balance"] < 0),
    ["_id", "financials.annual_income", "financials.savings_balance"]
]

,_id,financials.annual_income,financials.savings_balance
159,app_290,104000.0,-5000


Remove app_290 from df_clean and adding it to df_flagged:

In [1111]:
df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"] == "app_290"].assign(flag_reason="Negative savings balance")])
df_clean = df_clean[df_clean["_id"] != "app_290"]

C:\Users\fmdmn\AppData\Local\Temp\ipykernel_6220\1841008036.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"] == "app_290"].assign(flag_reason="Negative savings balance")])


In [1112]:
df_flagged.head()

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,flag_reason
16,app_101,NaN,Sandra Smith,sandra.smith99@icloud.com,937-72-8731,172.30.237.98,Female,1997-03-23,90255,55000,0,0.30,15364,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,464,0,0,527,0,0,0,0,0,0,0,SSN collision
92,app_088,NaN,Susan Martinez,susan.martinez83@protonmail.com,780-24-9300,172.23.224.13,F,1986-10-15,90229,55000,55,0.43,29118,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,339,0,0,SSN collision
122,app_016,NaN,Gary Wilson,gary.wilson85@yahoo.com,780-24-9300,192.168.20.42,M,1959-12-11,10091,123000,71,0.37,86599,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,485,0,0,0,0,0,0,0,SSN collision
499,app_234,NaN,Samuel Hill,samuel.hill67@protonmail.com,937-72-8731,10.143.146.157,Male,1976/01/29,10090,96000,60,0.30,38703,False,algorithm_risk_score,education,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,526,0,0,0,0,0,SSN collision
159,app_290,NaT,Stephanie Perez,stephanie.perez69@gmail.com,866-48-3726,172.25.130.219,Female,1990-01-06 00:00:00,90221,104000.0,39,0.27,-5000,True,NaN,NaN,4.3,49000.0,NaN,NaN,0,0,0,0,0,599,0,0,0,0,0,0,0,0,0,Negative savings balance


Checking whether there are applicants that have spendings total greater than their income:

In [1113]:
df_clean.loc[
    df_clean[[c for c in df_clean.columns if c.startswith("spend_")]].sum(axis=1) > df_clean["financials.annual_income"]
]

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities
426,app_190,2024-01-15 00:00:00+00:00,Heather White,heather.white42@icloud.com,217-44-8191,192.168.242.9,Female,1999-10-05,90297,0.0,0,0.14,19603,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,0,317,0,0,849,0,0,0,0,0,223,0,0,0


Extract and remove app_190 from df_clean

In [1114]:
df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"] == "app_190"].assign(flag_reason="Total spend exceeds annual income")])
df_clean = df_clean[df_clean["_id"] != "app_190"]

C:\Users\fmdmn\AppData\Local\Temp\ipykernel_6220\3402619509.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"] == "app_190"].assign(flag_reason="Total spend exceeds annual income")])


In [1115]:
df_flagged

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,flag_reason
16,app_101,NaT,Sandra Smith,sandra.smith99@icloud.com,937-72-8731,172.30.237.98,Female,1997-03-23,90255,55000,0,0.30,15364,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,464,0,0,527,0,0,0,0,0,0,0,SSN collision
92,app_088,NaT,Susan Martinez,susan.martinez83@protonmail.com,780-24-9300,172.23.224.13,F,1986-10-15,90229,55000,55,0.43,29118,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,339,0,0,SSN collision
122,app_016,NaT,Gary Wilson,gary.wilson85@yahoo.com,780-24-9300,192.168.20.42,M,1959-12-11,10091,123000,71,0.37,86599,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,485,0,0,0,0,0,0,0,SSN collision
499,app_234,NaT,Samuel Hill,samuel.hill67@protonmail.com,937-72-8731,10.143.146.157,Male,1976/01/29,10090,96000,60,0.30,38703,False,algorithm_risk_score,education,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,526,0,0,0,0,0,SSN collision
159,app_290,NaT,Stephanie Perez,stephanie.perez69@gmail.com,866-48-3726,172.25.130.219,Female,1990-01-06 00:00:00,90221,104000.0,39,0.27,-5000,True,NaN,NaN,4.3,49000.0,NaN,NaN,0,0,0,0,0,599,0,0,0,0,0,0,0,0,0,Negative savings balance
426,app_190,2024-01-15 00:00:00+00:00,Heather White,heather.white42@icloud.com,217-44-8191,192.168.242.9,Female,1999-10-05 00:00:00,90297,0.0,0,0.14,19603,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,317,0,0,849,0,0,0,0,0,223,0,0,0,Total spend exceeds annual income


Created an age column to check for impossible ages regarding credit decisions, must be over 18 and cant be over 100 years old:

In [1116]:
df_clean["age"] = (
    (pd.Timestamp.today() - df_clean["applicant_info.date_of_birth"])
    .dt.days // 365
)

In [1117]:
df_clean[
    (df_clean["age"] < 18) |
    (df_clean["age"] > 100)
]

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,age


Checking whether an applicant has had credit longer than it is legally possible:

In [1118]:
df_clean[
    (df_clean["financials.credit_history_months"] / 12) >
    (df_clean["age"] - 18)
][["_id", "age", "financials.credit_history_months"]]

,_id,age,financials.credit_history_months
496,app_049,25.0,92


In [1119]:
df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"] == "app_049"].assign(flag_reason="Credit history exceeds legal possible limit")])
df_clean = df_clean[df_clean["_id"] != "app_049"]

Checking whether credit history months is >= 0 :

In [1120]:
df_clean[
    (df_clean["financials.credit_history_months"] < 0) |
    (df_clean["financials.credit_history_months"] % 1 != 0)
][["_id","financials.credit_history_months"]]

,_id,financials.credit_history_months
137,app_043,-10
162,app_156,-3


In [1121]:
df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"].isin(["app_043", "app_156"])].assign(flag_reason="Invalid credit history (negative)")])
df_clean = df_clean[~df_clean["_id"].isin(["app_043", "app_156"])]

In [1122]:
df_flagged

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,flag_reason,age
16,app_101,NaT,Sandra Smith,sandra.smith99@icloud.com,937-72-8731,172.30.237.98,Female,1997-03-23,90255,55000,0,0.30,15364,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,464,0,0,527,0,0,0,0,0,0,0,SSN collision,NaN
92,app_088,NaT,Susan Martinez,susan.martinez83@protonmail.com,780-24-9300,172.23.224.13,F,1986-10-15,90229,55000,55,0.43,29118,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,339,0,0,SSN collision,NaN
122,app_016,NaT,Gary Wilson,gary.wilson85@yahoo.com,780-24-9300,192.168.20.42,M,1959-12-11,10091,123000,71,0.37,86599,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,485,0,0,0,0,0,0,0,SSN collision,NaN
499,app_234,NaT,Samuel Hill,samuel.hill67@protonmail.com,937-72-8731,10.143.146.157,Male,1976/01/29,10090,96000,60,0.30,38703,False,algorithm_risk_score,education,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,526,0,0,0,0,0,SSN collision,NaN
159,app_290,NaT,Stephanie Perez,stephanie.perez69@gmail.com,866-48-3726,172.25.130.219,Female,1990-01-06 00:00:00,90221,104000.0,39,0.27,-5000,True,NaN,NaN,4.3,49000.0,NaN,NaN,0,0,0,0,0,599,0,0,0,0,0,0,0,0,0,Negative savings balance,NaN
426,app_190,2024-01-15 00:00:00+00:00,Heather White,heather.white42@icloud.com,217-44-8191,192.168.242.9,Female,1999-10-05 00:00:00,90297,0.0,0,0.14,19603,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,317,0,0,849,0,0,0,0,0,223,0,0,0,Total spend exceeds annual income,NaN
496,app_049,NaT,Donna Gonzalez,donna.gonzalez12@yahoo.com,845-50-3562,192.168.14.85,Female,2000-05-22 00:00:00,90278,59000.0,92,0.36,31881,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,848,444,0,131,0,0,0,0,0,0,0,Credit history exceeds legal possible limit,25.0
137,app_043,NaT,Daniel King,daniel.king27@mail.com,166-23-2000,10.204.248.60,Male,1985-08-17 00:00:00,10038,131000.0,-10,0.06,53098,True,<NA>,NaN,3.9,68000.0,NaN,NaN,0,0,0,0,464,0,0,398,0,0,0,0,0,0,0,Invalid credit history (negative),40.0
162,app_156,2024-01-15 00:00:00+00:00,Jessica Green,jessica.green86@gmail.com,801-19-4595,192.168.59.201,Female,1999-11-28 00:00:00,90239,25000.0,-3,0.21,13641,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,218,0,0,0,0,Invalid credit history (negative),26.0


In [1123]:
df_clean[~df_clean["applicant_info.email"].str.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")]

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,age
138,app_204,NaT,Jonathan Carter,mike johnson@gmail.com,972-81-1928,10.77.72.174,Male,1983-04-12,90256,81000.0,63,0.22,14018,False,algorithm_risk_score,vacation,NaN,NaN,NaN,<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,778,0,42.0
181,app_299,NaT,Samuel Gonzalez,test.user.outlook.com,626-69-8549,172.29.212.150,Male,1972-08-28,10054,64000.0,58,0.18,23101,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,0,854,0,0,0,0,531,787,0,0,0,0,0,0,53.0
276,app_068,NaT,Emily Lopez,john.doe@invalid,641-36-7605,172.19.231.218,Female,1993-09-05,90282,75000.0,30,0.31,40791,True,<NA>,<NA>,4.9,20000.0,NaN,<NA>,0,0,0,0,0,0,0,0,577,0,0,0,0,0,188,32.0
369,app_146,2024-01-15 00:00:00+00:00,Amy Flores,sarah.smith@,577-59-1479,10.4.113.112,Female,1970-04-26,90262,79000.0,102,0.20,24961,True,<NA>,<NA>,2.7,61000.0,NaN,<NA>,0,0,0,687,0,0,0,0,0,0,0,73,0,0,0,55.0


Checking for impossible Debt to Income Ratio, the value should be between 0 and 1

In [1124]:
df_clean[
    (df_clean["financials.debt_to_income"] < 0) |
    (df_clean["financials.debt_to_income"] > 1)
]

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,age
316,app_402,NaT,Heather Flores,heather.flores23@mail.com,100-58-8097,172.21.176.42,Female,1965-03-07,90214,88000.0,27,1.85,37281,True,<NA>,<NA>,3.2,17000.0,NaN,<NA>,0,0,0,482,0,0,0,0,0,0,0,0,0,0,0,61.0


Adding app_402 to df_flagged and removing from df_clean

In [1125]:
df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"] == "app_402"].assign(flag_reason="Debt to income ratio out of valid range")])
df_clean = df_clean[df_clean["_id"] != "app_402"]

We found that there are two records with processing timestamps in the future, however the rest of the record seems to be clean so it probably is just a processing error, given that we will not move these 2 records to df_flagged.

In [1126]:
df_clean.loc[
    df_clean["processing_timestamp"] > pd.Timestamp.now(tz="UTC")
]

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,age
169,app_179,2026-03-15 00:00:00+00:00,Paul Roberts,paul.roberts46@yahoo.com,880-83-3129,172.21.233.185,Male,1993-07-12,10092,85000.0,30,0.24,31966,True,<NA>,<NA>,3.4,42000.0,NaN,<NA>,0,0,0,0,0,0,0,0,392,0,0,0,0,0,842,32.0
441,app_051,2027-01-20 00:00:00+00:00,Nicholas Thompson,nicholas.thompson68@icloud.com,700-28-7948,172.16.4.236,Male,1999-07-15,10060,35000.0,77,0.20,9646,False,low_income,<NA>,NaN,NaN,NaN,<NA>,0,0,251,0,0,0,0,0,0,0,0,0,0,0,0,26.0


Added to df_flagged but kept in df_clean to not bias analysis done in the future (created a column "kept_in_clean" that says whether the record is present in df_clean)

In [1127]:
df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"].isin(["app_179", "app_051"])].assign(flag_reason="Processing timestamp in the future", kept_in_clean=True)])


Checking for invalid email formats

In [1128]:
df_clean[~df_clean["applicant_info.email"].str.match(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")]

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes,spend_Adult Entertainment,spend_Alcohol,spend_Dining,spend_Education,spend_Entertainment,spend_Fitness,spend_Gambling,spend_Groceries,spend_Healthcare,spend_Insurance,spend_Rent,spend_Shopping,spend_Transportation,spend_Travel,spend_Utilities,age
138,app_204,NaT,Jonathan Carter,mike johnson@gmail.com,972-81-1928,10.77.72.174,Male,1983-04-12,90256,81000.0,63,0.22,14018,False,algorithm_risk_score,vacation,NaN,NaN,NaN,<NA>,0,0,0,0,0,0,0,0,0,0,0,0,0,778,0,42.0
181,app_299,NaT,Samuel Gonzalez,test.user.outlook.com,626-69-8549,172.29.212.150,Male,1972-08-28,10054,64000.0,58,0.18,23101,False,algorithm_risk_score,<NA>,NaN,NaN,NaN,<NA>,0,0,854,0,0,0,0,531,787,0,0,0,0,0,0,53.0
276,app_068,NaT,Emily Lopez,john.doe@invalid,641-36-7605,172.19.231.218,Female,1993-09-05,90282,75000.0,30,0.31,40791,True,<NA>,<NA>,4.9,20000.0,NaN,<NA>,0,0,0,0,0,0,0,0,577,0,0,0,0,0,188,32.0
369,app_146,2024-01-15 00:00:00+00:00,Amy Flores,sarah.smith@,577-59-1479,10.4.113.112,Female,1970-04-26,90262,79000.0,102,0.20,24961,True,<NA>,<NA>,2.7,61000.0,NaN,<NA>,0,0,0,687,0,0,0,0,0,0,0,73,0,0,0,55.0


Added to df_flagged but kept in df_clean to not bias analysis done in the future

In [1129]:
df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"].isin(["app_204", "app_299", "app_068", "app_146"])].assign(flag_reason="Invalid email format", kept_in_clean=True)])


Records missing core identity fields (full name, date of birth, or SSN) or annual income cannot be reliably identified, verified, or assessed for creditworthiness, making them unprocessable for credit analysis and therefore excluded from the analytical dataset.

In [1130]:
df_clean[df_clean[["applicant_info.full_name", "applicant_info.date_of_birth", "applicant_info.ssn", "financials.annual_income"]].isnull().any(axis=1)][["_id", "applicant_info.full_name", "applicant_info.date_of_birth", "applicant_info.ssn", "financials.annual_income"]]

,_id,applicant_info.full_name,applicant_info.date_of_birth,applicant_info.ssn,financials.annual_income
26,app_075,Margaret Williams,NaT,<NA>,61000.0
76,app_436,Amanda Torres,1998-06-02,135-51-1195,NaN
94,app_421,Donald Baker,1982-07-17,344-50-4287,NaN
99,app_479,Sandra Jones,1983-11-08,424-34-1670,NaN
141,app_463,Emma Clark,1975-04-26,976-47-3536,NaN
149,app_449,Lisa Roberts,1968-11-09,833-71-7935,NaN
275,app_120,Carolyn Martin,NaT,<NA>,103000.0
297,app_268,Larry Williams,1971-05-14,<NA>,112000.0
448,app_350,Linda Adams,NaT,356-98-8263,89000.0
462,app_165,Brandon Moore,NaT,<NA>,66000.0


Added these records to df_flagged and removed from df_clean

In [1131]:
df_flagged = pd.concat([df_flagged, df_clean[df_clean["_id"].isin(["app_075", "app_436", "app_421", "app_479", "app_463", "app_449", "app_120", "app_268", "app_350", "app_165"])].assign(flag_reason="Missing core identity fields", kept_in_clean=False)])
df_clean = df_clean[~df_clean["_id"].isin(["app_075", "app_436", "app_421", "app_479", "app_463", "app_449", "app_120", "app_268", "app_350", "app_165"])]

# Final Adjustment of Columns and Records in df_clean

Removing financials.annual_salary from both datasets since it is a copy of financials.annual_income but with 99% missing values

In [1132]:
df_clean = df_clean.drop(columns=["financials.annual_salary"], errors="ignore")
df_flagged = df_flagged.drop(columns=["financials.annual_salary"], errors="ignore")

Renaming the columns into a more user friendly name:

In [1133]:
df_clean.columns = (
    df_clean.columns
        .str.replace("applicant_info.", "", regex=False)
        .str.replace("financials.", "", regex=False)
        .str.replace("decision.", "", regex=False)
        .str.replace(".", "_", regex=False)
        .str.lower()
)

Changing "_id" to "application_id"

In [1134]:
df_clean = df_clean.rename(columns={"_id": "application_id"})

Renaming the columns in df_flagged the same as the ones in df_clean:

In [1135]:
df_flagged.columns = (
    df_flagged.columns
        .str.replace("applicant_info.", "", regex=False)
        .str.replace("financials.", "", regex=False)
        .str.replace("decision.", "", regex=False)
        .str.replace(".", "_", regex=False)
        .str.lower()
)

df_flagged = df_flagged.rename(columns={"_id": "application_id"})

In [1136]:
df_flagged["kept_in_clean"] = df_flagged["kept_in_clean"].fillna(False)

C:\Users\fmdmn\AppData\Local\Temp\ipykernel_6220\3091622527.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_flagged["kept_in_clean"] = df_flagged["kept_in_clean"].fillna(False)


Keeping only 3 columns that serve the purpose of df_flagged

In [1137]:
df_flagged = df_flagged[["application_id", "flag_reason", "kept_in_clean"]]

Removing "Notes" column from df_clean since it is 100% missing values:

In [1138]:
df_clean = df_clean.drop(columns=["notes"])

The original dataset contained 502 records. After removing 2 exact duplicate rows (confirmed as data entry errors via the notes column), we began the cleaning process on the remaining 500 records.

Two datasets were produced as a result of the cleaning process:

- **df_flagged** collects all records where a data quality issue was identified, along with the reason for flagging. Some of these records were kept in df_clean (e.g. invalid email format, future processing timestamps) since the issue was minor and would not affect bias analysis. Others were fully excluded.

- **df_clean** is the analytical dataset, containing 480 records (500 − 20 records with critical data quality issues). Excluded records had problems that made them unprocessable, such as SSN collisions, missing core identity fields, missing annual income, or invalid financial values.

### Saving df_clean and df_flagged to the data folder

In [1139]:
df_clean.to_csv("../data/df_clean.csv", index=False)
df_flagged.to_csv("../data/df_flagged.csv", index=False)
